In [1]:
# Immportations
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## Data train preparation

In [14]:
# Charger les données
df = pd.read_csv("/kaggle/input/datasets/ahmedsouleymanesow/bike-sharing/train.csv")

# Supprimer colonnes non utiles
df = df.drop(columns=["casual","registered"])

# Feature Engineering
df['datetime'] = pd.to_datetime(df['datetime'])
df['hour'] = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.weekday
df['month'] = df['datetime'].dt.month

# Colonnes catégorielles
cat_cols = ['season','holiday','workingday','weather','hour','weekday','month']
num_cols = ['temp','humidity','windspeed']

# One-hot encoder les colonnes catégorielles
ohe = OneHotEncoder(sparse_output=False)
cat_features = ohe.fit_transform(df[cat_cols])

# Normaliser les colonnes numériques
scaler = StandardScaler()
num_features = scaler.fit_transform(df[num_cols])

# Combiner
import numpy as np
X = np.hstack([num_features, cat_features])

# Target avec log-transform
y = np.log1p(df['count'].values).reshape(-1,1)


# Split train/val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Convertir en tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)


# Dataset personnalisé
class BikeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = BikeDataset(X_train, y_train)
val_dataset = BikeDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

## Model

In [13]:
# Modèle MLP 
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=X_train.shape[1])

# Loss & Optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Trainig Model

In [15]:
# Training
epochs = 50

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")
torch.save(model.state_dict(), 'baseline_lin_model.pth')


Epoch 1/50 - Loss: 4.9188
Epoch 2/50 - Loss: 0.7533
Epoch 3/50 - Loss: 0.6162
Epoch 4/50 - Loss: 0.4889
Epoch 5/50 - Loss: 0.4337
Epoch 6/50 - Loss: 0.4247
Epoch 7/50 - Loss: 0.4189
Epoch 8/50 - Loss: 0.4164
Epoch 9/50 - Loss: 0.4015
Epoch 10/50 - Loss: 0.4045
Epoch 11/50 - Loss: 0.4039
Epoch 12/50 - Loss: 0.3934
Epoch 13/50 - Loss: 0.3778
Epoch 14/50 - Loss: 0.3778
Epoch 15/50 - Loss: 0.3716
Epoch 16/50 - Loss: 0.3625
Epoch 17/50 - Loss: 0.3737
Epoch 18/50 - Loss: 0.3498
Epoch 19/50 - Loss: 0.3585
Epoch 20/50 - Loss: 0.3419
Epoch 21/50 - Loss: 0.3564
Epoch 22/50 - Loss: 0.3379
Epoch 23/50 - Loss: 0.3357
Epoch 24/50 - Loss: 0.3358
Epoch 25/50 - Loss: 0.3320
Epoch 26/50 - Loss: 0.3289
Epoch 27/50 - Loss: 0.3307
Epoch 28/50 - Loss: 0.3115
Epoch 29/50 - Loss: 0.3168
Epoch 30/50 - Loss: 0.3060
Epoch 31/50 - Loss: 0.2930
Epoch 32/50 - Loss: 0.3056
Epoch 33/50 - Loss: 0.2968
Epoch 34/50 - Loss: 0.2909
Epoch 35/50 - Loss: 0.2867
Epoch 36/50 - Loss: 0.2799
Epoch 37/50 - Loss: 0.2752
Epoch 38/5

## Validation

In [16]:
model.load_state_dict(torch.load('baseline_lin_model.pth'))
model.eval()
with torch.no_grad():
    preds_log = model(X_val)
    preds = torch.expm1(preds_log)  # revenir à l’échelle originale
    y_true = torch.expm1(y_val)
    rmse = torch.sqrt(nn.MSELoss()(preds, y_true))

print("\nValidation RMSE:", rmse.item())


Validation RMSE: 75.65685272216797


## Testing

### Test data preparation

In [17]:
# Charger test
df_test = pd.read_csv("/kaggle/input/datasets/ahmedsouleymanesow/bike-sharing/test.csv")

# Feature engineering
df_test['datetime'] = pd.to_datetime(df_test['datetime'])
df_test['hour'] = df_test['datetime'].dt.hour
df_test['weekday'] = df_test['datetime'].dt.weekday
df_test['month'] = df_test['datetime'].dt.month

# Transformer catégorielles (PAS FIT)
cat_features_test = ohe.transform(df_test[cat_cols])

# Transformer numériques (PAS FIT)
num_features_test = scaler.transform(df_test[num_cols])

# Combiner
X_test = np.hstack([num_features_test, cat_features_test])

# Convertir en tensor
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)


In [18]:
df_test.shape

(6493, 12)

### Test

In [19]:
model.load_state_dict(torch.load('baseline_lin_model.pth'))
model.eval()

with torch.no_grad():
    predictions = model(X_test_tensor)

predictions = predictions.numpy()

# Inverser log-transform
predictions = np.expm1(predictions)

# Aplatir
predictions = predictions.flatten()

### Saving the predictions

In [20]:
submission = pd.DataFrame({
    "datetime": df_test["datetime"],
    "count": predictions
})
submission["count"] = submission["count"].round().astype(int)
submission.to_csv("/kaggle/working/submission_ah_sk.csv", index=False)

In [21]:
df = pd.read_csv('/kaggle/working/submission_ah_sk.csv')
df.head()

,datetime,count
0,2011-01-20 00:00:00,18
1,2011-01-20 01:00:00,7
2,2011-01-20 02:00:00,3
3,2011-01-20 03:00:00,3
4,2011-01-20 04:00:00,3


In [22]:
df.shape

(6493, 2)